# Datamine COPYNR Process Example

This notebook demonstrates how to configure and run the **`copynr`** process wrapper in `dmstudio`.

## Process Description

## COPYNR

Copies a database file to a new file with the addition of a field for the record (line) number. This field is named **RECORDNO**.

By default the record numbers start at 1 and increment in steps of 1, but the start number and increment may optionally be chosen by entering required values for the parameters @**BASE** and @**INCRMENT**. If **RECORDNO** already exists, it will be renumbered. Thus **COPYNR** acts as both a number and a re-number facility.

With retrieval criteria, the output file will contain those records that match the criteria; record numbers will start at @BASE with increment @**INCRMENT** on these output records.

If the @**BASE** and @**INCRMENT** parameters entered are not integers, then they will be truncated to integers. For example, an @INCRMENT of 0.5 will be truncated to 0, leading to all records being given the same **RECORDNO**.

No check is made for the existence of the specified output file, which therefore can be overwritten.

### Input Files:

* **in** (Table):
  Input file.
  Required=Yes

### Output Files:

* **out** (Table):
  Output file; same as input with addition of field **RECORDNO**. If **RECORDNO** already
  exists in the input file, values will be renumbered in the output.
  Required=Yes

### Fields:

### Parameters:

* **base**:
  Starting record number (1) [Integer].
  Range=
  Values=Undefined
  Default=
  Required=No

* **incrment**:
  Record number increment (1) [Integer].
  Range=
  Values=Undefined
  Default=
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('copynr')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute copynr
print("Running copynr...")
dm_cmd.copynr(
    in_i='t_assays',  # required
    out_o='t_copynr_out',  # required
    # base_p='optional',  # optional
    # incrment_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("copynr execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_copynr_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")